In [15]:
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from typing import Optional, TypedDict, Annotated
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [6]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview", 
    output_dimensionality=768
)

In [13]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [11]:
file_path = "book.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, separators=["\n\n", "\n", " ", ""])
chunks = text_splitter.split_documents(documents)

vectorstore = FAISS.from_documents(chunks, embeddings)

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [4]:
class State(TypedDict):
    question:str
    docs:list[Document]

    strips: list[str]
    kept_strips: list[str]
    refined_context: str

    answer:str

In [10]:
def retrieve(state: State):
    """
    Retrieve relevant documents based on the question.
    """
    query = state["question"]
    retrieved_docs = retriever.invoke(query)
    return {"docs": retrieved_docs}
    

In [12]:
import re
def decompose_to_strips(text:str)->list[str]:
    text = re.sub(r'\s+', ' ', text).strip()
    sentences = re.split(r'(?<=[.!?]) +', text)
    return [s.strip() for s in sentences if len(s.strip())>20]

In [16]:
from pydantic import BaseModel, Field
class KeepStrip(BaseModel):
    keep: bool = Field(..., description="Whether to keep the strip or not")


filter_prompt= ChatPromptTemplate.from_messages([
    (
        "system",
        """"You are a strict relevance filter.\n
        You will be given a list of text strips and a question.\n"
        "Return keep=true if the sentence is relevant to the question, otherwise return keep=false.\n"
        Use only the sentence and output JSON only
        """
    ),
    (
        "human",
        """Question: {question}\n
        Sentence: {sentence}\n
        Output JSON only, no explanations.
        """
    )
])

llm_filter = llm.with_structured_output(KeepStrip)
filter_chain = filter_prompt | llm_filter

In [17]:
# REFINING (Decompose -> Filter -> Recompose)
def refine_context(state: State):
    """
    Refine the context by decomposing the retrieved documents into strips, filtering them based on relevance to the question, and recomposing the relevant strips into a refined context.
    """
    context = "\n\n".join([doc.page_content for doc in state["docs"]])
    strips = decompose_to_strips(context)
    filtered_strips = [strip for strip in strips if filter_chain.invoke({"question": state["question"], "sentence": strip}).keep]

    refined_context = "\n".join(filtered_strips)

    return {
        "strips": strips,
        "kept_strips": filtered_strips,
        "refined_context": refined_context
    }